# g1_limpo — SONDA: 2000 iterações sobre o `model_14000`

Não é bloco de treino. É um teste: as correções de 14/09 mudam o comportamento do robô
que já existe?

## O que está sendo testado

| mudança | comportamento que ela ataca |
|---|---|
| `velocidade_por_regime`: `mean` -> `amax` | velocidade de manipulação exagerada |
| `apoiada` vira faixa (teto `peso + 30 N`) | ele se apoia na caixa para botar |
| `upright` na tabela, BOTAR = 4 | idem — paga por tronco ereto onde ele deita |
| `faixa_de_pose.perna` no BOTAR = 0 | libera o agachamento, que é a alternativa a deitar |
| `faixa_de_pose` na tabela, BOTAR = 2 | mãos tortas no botar |

## O que está PINADO, e por quê

`PINA_ALVO = True` na célula do treino congela a altura do alvo em **1,02** — o valor
com que este checkpoint foi treinado.

⚠ A randomização 0,75 a 1,0 é **fora da distribuição** para esta política: ela nunca
viu alvo abaixo de 1,02. Ligá-la aqui misturaria "as correções funcionam?" com "ele
aprende uma altura nova em 2000 iterações?", e um resultado ruim não diria qual das
duas falhou. A randomização entra no treino DO ZERO.

Para testar a randomização em vez das correções: `PINA_ALVO = False`.

## O que olhar no painel

| canal | esperado |
|---|---|
| `Episode_Reward/velocidade_por_regime` | salta de ~-2,6 para muito mais negativo no primeiro passo, e depois SOBE |
| `Episode_Reward/faixa_de_pose` | idem: pior no começo, melhorando |
| `Episode_Reward/upright` | sobe |
| `Metrics/sucesso` e `s_C` | ⚠ CAEM no transiente. Não julgue antes de +200 iterações |
| `Episode_Termination/caixa_largada` | se disparar, o teto do `apoiada` está apertado demais |

⚠ O risco nomeado: se o BOTAR parar de fechar, `folga_apoiada_N = 30` é o suspeito.
A métrica `impacto_da_caixa` dá o pico de força por episódio — é ela que diz.

**Antes de rodar:** GPU, Internet On, os dois segredos anexados, a branch
`exp/g1-limpo-v2` **no GitHub**, e o dataset com o `model_14000` em Add Input.


In [ ]:
# ⚠ NADA DE `import torch` AQUI. O torch registra operadores C++ no import, e se ele
# entrar no kernel ANTES do pip, um reload depois levanta
# `Only a single TORCH_LIBRARY can be used to register the namespace triton`.
import subprocess, sys

smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                      "--format=csv,noheader"], capture_output=True, text=True)
print(smi.stdout or smi.stderr)
assert smi.returncode == 0 and smi.stdout.strip(), \
    "sem GPU. Settings -> Accelerator -> GPU"
print("python", sys.version.split()[0])

In [ ]:
import subprocess, sys

# ⚠⚠ LISTA DE ARGUMENTOS, NUNCA STRING DE SHELL. Com `!pip install ... numpy<2.5` o
# shell lê `<` como REDIRECIONAMENTO, tenta abrir um arquivo chamado `2.5`, aborta com
# exit 2 — e o pip NUNCA RODA, em silêncio.
cmd = [sys.executable, "-m", "pip", "install", "--no-warn-conflicts", "mjlab==1.5.3"]
print(" ".join(cmd), flush=True)
r = subprocess.run(cmd, capture_output=True, text=True)
print(r.stdout[-2000:])
if r.returncode != 0:
    print(r.stderr[-2000:])
assert r.returncode == 0, \
    "o pip falhou. Causa mais comum: internet DESLIGADA (Settings -> Internet -> On)"

In [ ]:
import subprocess, sys

chk = subprocess.run(
    [sys.executable, "-c",
     "import torch;print(torch.__version__, torch.cuda.is_available())"],
    capture_output=True, text=True)
print("subprocesso:", chk.stdout.strip() or chk.stderr[-600:])
assert " True" in chk.stdout, \
    "o pip trocou o torch e a CUDA foi embora. Reinicie o kernel e rode da célula 1"

import torch, mjlab, mujoco, warp
print(f"torch {torch.__version__}  {torch.cuda.get_device_name(0)}")
print("mjlab", getattr(mjlab, "__version__", "?"),
      "| mujoco", mujoco.__version__, "| warp", warp.config.version)

In [ ]:
import importlib, os, pathlib, re, shutil, subprocess, sys

os.environ.setdefault("MUJOCO_GL", "egl")

RUN       = "bloco19"              # ⚠ NOME NOVO. Ele separa o log desta sonda do log da
                                   # `bloco17`: a tabela de recompensa mudou, e misturar
                                   # as duas no mesmo `tfevents` torna a leitura inútil.
IT_MINIMA = 13000                  # trava contra checkpoint velho no dataset
BRANCH    = "exp/g1-limpo-v2"

BASE     = pathlib.Path("/kaggle/working")
RAIZ     = BASE / "g1"             # o clone, refeito a cada sessão
LOG_ROOT = BASE / "logs"           # ⚠ FORA de RAIZ: o re-clone apaga o que está dentro
raiz_exp = LOG_ROOT / "g1_limpo"

if RAIZ.exists():
    shutil.rmtree(RAIZ)
subprocess.run(["git", "clone", "-q", "--branch", BRANCH, "--depth", "1",
                "https://github.com/JoaoBornelli/g1_training.git", str(RAIZ)],
               check=True)
print("clone =", subprocess.run(["git", "-C", str(RAIZ), "log", "--oneline", "-1"],
                                capture_output=True, text=True).stdout.strip())

# ⚠ `invalidate_caches` NÃO é higiene. O Python cacheia um finder POR DIRETÓRIO, e o
# de um diretório que não existia na hora da inserção fica cacheado como VAZIO —
# `import g1_limpo` falharia com `No module named` mesmo com o pacote em disco.
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
importlib.invalidate_caches()

# ------------------------------- achar o checkpoint e semear a árvore do mjlab
# ⚠ PELO MAIOR NÚMERO, e não por nome fixo: o run fecha em `model_N` ou `model_N+1`
# conforme o `learn()` salvar o final, e adivinhar já custou uma rodada.
# ⚠ `rglob`: a Kaggle monta ora em `/kaggle/input/<slug>/`, ora em
# `/kaggle/input/datasets/<usuario>/<slug>/`.
entrada = pathlib.Path("/kaggle/input")
achados = sorted(entrada.rglob("model_*.pt"),
                 key=lambda q: int(re.search(r"model_(\d+)", q.stem).group(1)),
                 reverse=True)
assert achados, f"nenhum `model_*.pt` em {entrada}. Add Input -> G1-Limpo-V2"
origem = achados[0]
it_origem = int(re.search(r"model_(\d+)", origem.stem).group(1))
assert it_origem >= IT_MINIMA, \
    f"{origem.name} está na iteração {it_origem}, abaixo de {IT_MINIMA}: versão velha"

# ⚠ O `get_checkpoint_path` resolve `<log_root>/<experiment_name>/<run_dir>/<ckpt>`, e a
# pasta do dataset é PLANA — sem esta semeadura o resume levanta
# `No run directories found`.
# ⚠ E a pasta começa com 1900 DE PROPÓSITO: o `get_checkpoint_path` ordena os nomes e
# pega o último, portanto qualquer pasta real desta sessão (`2026-...`) ordena acima.
semente = raiz_exp / f"1900-01-01_00-00-00_{RUN}"
semente.mkdir(parents=True, exist_ok=True)
alvo = semente / origem.name
if not alvo.exists():
    shutil.copy2(origem, alvo)

print(f"origem  = {origem.name}  (it {it_origem}, "
      f"{origem.stat().st_size / 2**20:.1f} MB)")
print(f"semente = {alvo}")
print(f"run     = {RUN}")

In [ ]:
import pathlib, re, zipfile
from IPython.display import FileLink, display

SAIDA = BASE
ULTIMO_CKPT = ULTIMA_IT = ULTIMO_PACOTE = None


def _run_nova(run=RUN):
    """A pasta de run REAL mais recente e seus `model_*.pt` — nunca a semente 1900."""
    if not raiz_exp.is_dir():
        return None, []
    runs = sorted(p for p in raiz_exp.iterdir()
                  if p.is_dir() and p.name.endswith(run)
                  and not p.name.startswith("1900"))
    if not runs:
        return None, []
    cks = sorted(runs[-1].glob("model_*.pt"),
                 key=lambda p: int(re.search(r"(\d+)", p.name).group(1)))
    return runs[-1], cks


def empacota(run=RUN, baixa=True):
    """Zipa o ÚLTIMO checkpoint + os tfevents e oferece o download.

    ⚠ O tfevents vai junto: sem ele o `leitura.py` não tem o que ler, e é a LEITURA
    que decide se esta sonda deu certo.
    """
    global ULTIMO_CKPT, ULTIMA_IT, ULTIMO_PACOTE
    nova, cks = _run_nova(run)
    if not cks:
        print("nenhum checkpoint novo — o treino não salvou nada")
        return None
    ultimo = cks[-1]
    it = int(re.search(r"(\d+)", ultimo.name).group(1))
    pacote = SAIDA / f"{run}_it{it}.zip"
    with zipfile.ZipFile(pacote, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(ultimo, ultimo.name)
        for ev in sorted(nova.glob("events.out.tfevents*")):
            z.write(ev, ev.name)
        for p in sorted(nova.glob("params/*")):
            z.write(p, f"params/{p.name}")
    ULTIMO_CKPT, ULTIMA_IT, ULTIMO_PACOTE = ultimo, it, pacote
    print(f"{pacote}  ({pacote.stat().st_size / 2**20:.1f} MB)  it {it}")
    if baixa:
        display(FileLink(str(pacote.relative_to(BASE))))
    return pacote

In [ ]:
# =====================================================================
#  SONDA — 2000 iterações sobre o checkpoint, com as correções de 14/09
# =====================================================================
import dataclasses, sys
import torch
sys.path.insert(0, str(RAIZ))

import g1_limpo
from mjlab.scripts.train import TrainConfig, launch_training
from mjlab.utils.os import get_checkpoint_path

NUM_ENVS = 4096          # o número da Kaggle. Se der OOM: 2048.
ITERACOES = 2000         # quantas ACRESCENTAR ao checkpoint

# ⚠⚠ PINA O ALVO NA ALTURA COM QUE ESTE CHECKPOINT FOI TREINADO. A faixa 0,75–1,0 é
# FORA DA DISTRIBUIÇÃO para esta política — ela nunca viu alvo abaixo de 1,02. Com a
# randomização ligada, um resultado ruim não diria se falharam as correções ou se ele
# só não aprendeu a altura nova em 2000 iterações. Ponha `False` para testar a
# randomização em vez das correções.
PINA_ALVO = True
ALTURA_PINADA = 1.02

# ⚠ A Kaggle corta em 12 h. A 7,0 s/iter, 2000 iterações são ~3,9 h — cabe folgado.
HORAS_LIMITE = 10.5
SEG_POR_ITER = 7.0

cfg = dataclasses.replace(TrainConfig.from_task(g1_limpo.TASK_ID),
                          log_root=str(LOG_ROOT))
cfg.env.scene.num_envs = NUM_ENVS
cfg.agent.run_name = RUN
cfg.agent.logger = "tensorboard"

# ------------------------------------------------------------------ O RESUME
cfg.agent.resume = True
# ⚠ `load_run` PINADO. O default `.*` casa QUALQUER run do experimento, e o
# `get_checkpoint_path` pega a de nome mais ALTO em ordem alfabética.
cfg.agent.load_run = f".*_{RUN}$"
cfg.agent.load_checkpoint = r"model_\d+\.pt"

# ⚠⚠ ISTO É WARM-START, E NÃO RESUME. A tabela de recompensa mudou: o
# `velocidade_por_regime` passou de `mean` para `amax`, o `upright` e o
# `faixa_de_pose` entraram na tabela por estado, e o fecho do BOTAR ganhou teto de
# força. A função de valor está errada por vários pontos por segundo.
# O degrau para 5e-4 é o que a nota `warmstart-lower-lr` manda fazer. O transiente
# dura 120 a 150 iterações — não julgue esta sonda antes de +200.
cfg.agent.algorithm.learning_rate = 5.0e-4

if PINA_ALVO:
    _c = cfg.env.commands["alvo_caixa"]
    _c.altura_carregar = ALTURA_PINADA
    _c.altura_carregar_faixa = (ALTURA_PINADA, ALTURA_PINADA)

# ----------------------------------------- ler o checkpoint ANTES de lançar
ckpt = get_checkpoint_path(raiz_exp.resolve(), cfg.agent.load_run,
                           cfg.agent.load_checkpoint)
it0 = int(torch.load(ckpt, map_location="cpu", weights_only=True)["iter"])

# ⚠⚠ `max_iterations` É ADITIVO. `rsl_rl/runners/on_policy_runner.py:78` faz
#     total_it = current_learning_iteration + num_learning_iterations
cabe_tempo = int(HORAS_LIMITE * 3600 / SEG_POR_ITER)
cfg.agent.max_iterations = min(ITERACOES, cabe_tempo)

_c = cfg.env.commands["alvo_caixa"]
print(f"checkpoint  = {ckpt.name}  (it {it0})")
print(f"vai rodar   = {cfg.agent.max_iterations} -> termina na "
      f"{it0 + cfg.agent.max_iterations}")
print(f"envs        = {NUM_ENVS}   lr {cfg.agent.algorithm.learning_rate} "
      f"({cfg.agent.algorithm.schedule})   seed {cfg.agent.seed}")
print(f"alvo z      = {_c.altura_carregar_faixa}"
      + ("   [PINADO]" if PINA_ALVO else "   [SORTEADO — fora da distribuição]"))
print(f"folga_apoiada_N = {_c.folga_apoiada_N} N   "
      f"teto do `apoiada` = m·g + {_c.folga_apoiada_N}")
print(f"log_root    = {cfg.log_root}\n")

# ⚠ `try/finally`, e não a linha nua. Assim o pacote nasce também quando o treino
# estoura ou quando você interrompe o kernel.
try:
    launch_training(g1_limpo.TASK_ID, cfg)
finally:
    empacota()

## Persistir o checkpoint fora da sessão

Sobe como **versão nova do dataset**. É a única persistência: `/kaggle/working` morre
com a sessão.

⚠ Esta sonda produz um checkpoint de uma tabela de recompensa NOVA. Subi-lo por cima
do `model_14000` no mesmo dataset faz o notebook de resume achar este, e não aquele —
o que é o certo se a sonda der bom, e errado se você quiser voltar. **Baixe o zip
antes**, pelo link que a célula do treino imprime.


In [ ]:
import json, os, pathlib, shutil, subprocess

if ULTIMO_CKPT is None:
    empacota(baixa=False)
assert ULTIMO_CKPT is not None, "nada para subir: o treino não salvou checkpoint"

SLUG = "g1-limpo-v2"        # o slug do SEU dataset, sem o nome de usuário

from kaggle_secrets import UserSecretsClient
_s = UserSecretsClient()
os.environ["KAGGLE_USERNAME"] = _s.get_secret("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = _s.get_secret("KAGGLE_KEY")
usuario = os.environ["KAGGLE_USERNAME"]

# ⚠ PASTA PRÓPRIA, com SÓ o que sobe. O `kaggle datasets version` envia o diretório
# INTEIRO — apontá-lo para /kaggle/working mandaria o clone e todos os checkpoints.
envio = pathlib.Path("/kaggle/working/envio")
shutil.rmtree(envio, ignore_errors=True)
envio.mkdir()
shutil.copy2(ULTIMO_CKPT, envio / ULTIMO_CKPT.name)
(envio / "dataset-metadata.json").write_text(json.dumps(
    {"title": "G1-Limpo-V2", "id": f"{usuario}/{SLUG}",
     "licenses": [{"name": "CC0-1.0"}]}, indent=1))
print("vai subir:", sorted(p.name for p in envio.iterdir()))

r = subprocess.run(["kaggle", "datasets", "version", "-p", str(envio),
                    "-m", f"{RUN} it{ULTIMA_IT}", "-r", "skip"],
                   capture_output=True, text=True)
print(r.stdout or r.stderr)
assert r.returncode == 0, (
    "o upload falhou. Confira: os dois segredos anexados, a internet ligada, e o "
    f"dataset {usuario}/{SLUG} existindo e sendo seu.")
print(f"\nsubiu {ULTIMO_CKPT.name} em {usuario}/{SLUG}")